# 02 - Data Preprocessing

This notebook converts the raw AI4I data into leakage-safe training and testing matrices. The current run selects **six input features**, creates an **8,000-record training set** and a **2,000-record test set**, one-hot encodes machine type, and standardizes the five numerical sensor features using training-set statistics only.

In [3]:
# Importing the basic libraries for data manipulation and preprocessing

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

In [7]:
# Load the dataset
df = pd.read_csv('../../data/raw/ai4i2020.csv')


## Step 1: Check for Missing Values

Missing values must be identified before training because many machine-learning estimators cannot process them directly.

In [8]:
print("Missing values per column:")
print(df.isna().sum())
print("\n" + "="*60 + "\n")

Missing values per column:
UID                    0
Product ID             0
Type                   0
Air temperature        0
Process temperature    0
Rotational speed       0
Torque                 0
Tool wear              0
Machine failure        0
TWF                    0
HDF                    0
PWF                    0
OSF                    0
RNF                    0
dtype: int64




**Observed result:** every column has zero missing values, so no rows are removed and no values are imputed. This preserves all 10,000 records.

In [9]:
# Step 2: Extract product quality variant from Product ID
# Product ID format: e.g., "L50001" -> quality = "L", serial = "50001"
df["product_quality"] = df["Product ID"].str[0]  # Extract first letter
print("Unique product quality variants:")
print(df["product_quality"].unique())
print("Product quality distribution:")
print(df["product_quality"].value_counts())


Unique product quality variants:
<StringArray>
['M', 'L', 'H']
Length: 3, dtype: str
Product quality distribution:
product_quality
L    6000
M    2997
H    1003
Name: count, dtype: int64


## Step 2: Examine Product Quality

The extracted quality values contain **6,000 L**, **2,997 M**, and **1,003 H** records. A direct comparison confirms that `product_quality` is identical to the existing `Type` column for all 10,000 records. It is therefore redundant and is not added to the model inputs.

Machine type may still contain useful operating information, so the original `Type` column is retained as the single categorical input. Keeping only one representation avoids duplicated information and makes later explanations clearer.

## Step 3: Check Target Balance

The target contains **9,661 no-failure records** and **339 failure records**, corresponding to a 3.39% failure rate.

In [10]:

print("Target variable balance (before split):")
print(df["Machine failure"].value_counts())
print(f"Failure rate: {df['Machine failure'].mean():.2%}")

Target variable balance (before split):
Machine failure
0    9661
1     339
Name: count, dtype: int64
Failure rate: 3.39%


Because only 3.39% of records represent failure, accuracy alone would be misleading. A constant no-failure prediction would achieve 96.61% accuracy but zero failure recall. The later classifier therefore uses balanced class weights and is evaluated using precision, recall, F1-score, and a confusion matrix. Resampling is a possible alternative, but it is not applied in this preprocessing experiment.

## Step 4: Prepare Features and Target

In [ ]:

input_features = [
    "Type",
    "Air temperature",
    "Process temperature",
    "Rotational speed",
    "Torque",
    "Tool wear",
]

label_columns = ["Machine failure", "TWF", "HDF", "PWF", "OSF", "RNF"]

X = df[input_features].copy()
y = df["Machine failure"].copy()

failure_modes = df[["TWF", "HDF", "PWF", "OSF", "RNF"]].copy()

print("Input feature columns:")
print(X.columns.tolist())

print("\nTarget column:")
print(y.name)

print("\nFailure mode columns kept only for analysis/evaluation:")
print(failure_modes.columns.tolist())

print("\nX shape:", X.shape)
print("y shape:", y.shape)

Input feature columns:
['Type', 'Air temperature', 'Process temperature', 'Rotational speed', 'Torque', 'Tool wear']

Target column:
Machine failure

Failure mode columns kept only for analysis/evaluation:
['TWF', 'HDF', 'PWF', 'OSF', 'RNF']

X shape: (10000, 6)
y shape: (10000,)


The feature matrix has shape **(10,000, 6)** and contains `Type` plus five sensor readings. The target vector has shape **(10,000,)**. The failure-mode indicators (`TWF`, `HDF`, `PWF`, `OSF`, and `RNF`) are stored separately for post-model evaluation and are not input features because that would introduce target leakage.


## Step 5: Create a Stratified Train/Test Split

`stratify=y` preserves the rare failure proportion across both partitions. The resulting training set contains **7,729 normal records and 271 failures (3.3875%)**. The test set contains **1,932 normal records and 68 failures (3.40%)**. This gives the model enough failure examples for training while keeping an independent and representative evaluation set.

In [17]:
# Step 5: Train/test split before encoding and scaling
# Stratification keeps failure/no-failure proportions consistent in both sets

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2, # 20% test set and 80% training set
    random_state=42, # for randomness and reproducibility to make the same records enter the training and testing sets every time.
    stratify=y # to maintain the same failure/no-failure ratio in both sets
)

print(f"Training set size: {X_train_raw.shape[0]} samples")
print(f"Test set size: {X_test_raw.shape[0]} samples")
print(f"\nTraining set failure rate: {y_train.mean():.2%}")
print(f"Test set failure rate: {y_test.mean():.2%}")

Training set size: 8000 samples
Test set size: 2000 samples

Training set failure rate: 3.39%
Test set failure rate: 3.40%


## Step 6: Encode the Categorical Feature

One-hot encoding converts `Type` into `Type_L` and `Type_M`; type H is the reference category represented by zero in both columns. After encoding, the matrices contain **seven columns**: five numerical sensors plus two type indicators. Both training and testing matrices are explicitly aligned to the same column order.

In [21]:


X_train_encoded = pd.get_dummies(X_train_raw, columns=["Type"], drop_first=True, dtype=int)
X_test_encoded = pd.get_dummies(X_test_raw, columns=["Type"], drop_first=True, dtype=int)

# Make sure train and test have the same columns
X_test_encoded = X_test_encoded.reindex(columns=X_train_encoded.columns, fill_value=0)

print("Encoded training columns:")
print(X_train_encoded.columns.tolist())

print("Encoded testing columns:")
print(X_test_encoded.columns.tolist())

print("\nTrain shape:", X_train_encoded.shape)
print("Test shape:", X_test_encoded.shape)

Encoded training columns:
['Air temperature', 'Process temperature', 'Rotational speed', 'Torque', 'Tool wear', 'Type_L', 'Type_M']
Encoded testing columns:
['Air temperature', 'Process temperature', 'Rotational speed', 'Torque', 'Tool wear', 'Type_L', 'Type_M']

Train shape: (8000, 7)
Test shape: (2000, 7)


## Step 7: Scale Numerical Sensor Features

Standardization transforms each numerical value using $z=(x-\mu_{train})/\sigma_{train}$. Fitting the scaler only on training data prevents test-set information from leaking into model development.

### `fit_transform()` Versus `transform()`

`fit_transform(X_train)` learns the training means and standard deviations and then transforms the training records. `transform(X_test)` reuses those learned statistics. Fitting a new scaler on the test set would leak evaluation information.

After scaling, all five training sensor columns have means numerically equal to 0 and sample standard deviations of approximately 1.00006. The test-set means remain close to zero but are not exactly zero, which is expected because the test records are transformed using training statistics.

In [19]:


numeric_features = [
    "Air temperature",
    "Process temperature",
    "Rotational speed",
    "Torque",
    "Tool wear",
]

# Initialize the StandardScaler to scale numeric features of both test and train sets to have mean~0 and std=1
scaler = StandardScaler()

X_train_processed = X_train_encoded.copy()
X_test_processed = X_test_encoded.copy()

X_train_processed[numeric_features] = scaler.fit_transform(
    X_train_encoded[numeric_features]
)

X_test_processed[numeric_features] = scaler.transform(
    X_test_encoded[numeric_features]
)

print("Processed training data:")
print(X_train_processed.head())

print("\nProcessed test data:")
print(X_test_processed.head())

print("\n Train data mean and std after scaling:")
print(X_train_processed[numeric_features].mean())
print("\n")
print(X_train_processed[numeric_features].std())

print("\n Test data mean and std after scaling:")
print(X_test_processed[numeric_features].mean())
print("\n")
print(X_test_processed[numeric_features].std())

Processed training data:
      Air temperature  Process temperature  Rotational speed    Torque  \
4058         0.998914             0.604282         -0.460607  0.718305   
1221        -1.505194            -1.153260         -0.775574  0.638456   
6895         0.498092             1.077466         -1.007654  0.558607   
9863        -0.553633            -0.139294         -0.709265  1.626586   
8711        -1.455112            -1.018064          1.070019 -1.128202   

      Tool wear  Type_L  Type_M  
4058  -0.843997       0       1  
1221   0.382263       0       1  
6895   0.460870       0       1  
9863  -0.372359       1       0  
8711  -0.906882       1       0  

Processed test data:
      Air temperature  Process temperature  Rotational speed    Torque  \
2997         0.247681            -0.139294         -1.073963  2.265378   
4871         1.850310             1.618248         -0.145641  0.009646   
3858         1.249324             0.942270          0.108543 -0.239882   
951     

### Preprocessing Result

The final processed training matrix has shape **(8,000, 7)** and the processed test matrix has shape **(2,000, 7)**. The categorical columns remain binary, while the numerical columns are on a common standardized scale. The key reproducibility rule is: **anything that learns information from data must be fitted using the training set only.**